In [8]:
import polars as pl
from torch.utils.data import DataLoader, TensorDataset, random_split
from pathlib import Path
import numpy as np

In [2]:
cwd = Path.cwd().parent
df = pl.read_csv(cwd/'data/processed/CSI300_1d.csv')

In [3]:
df.head()

timestamp,open,high,low,close,volume,rv
str,f64,f64,f64,f64,f64,f64
"""2010-01-05T00:00:00.000000""",3592.47,3597.46,3535.23,3535.23,6.6101e9,0.001254
"""2010-01-06T00:00:00.000000""",3545.79,3577.34,3497.66,3564.04,8.5611e9,0.00198
"""2010-01-07T00:00:00.000000""",3556.98,3588.77,3541.28,3541.73,7.8250e9,0.001687
"""2010-01-08T00:00:00.000000""",3542.44,3558.56,3453.4,3471.46,8.0187e9,0.002066
"""2010-01-09T00:00:00.000000""",3453.45,3482.04,3426.82,3480.13,6.0607e9,0.002053


In [4]:
df = df.with_columns(
    df.sql('''
       SELECT 
       SUBSTR(timestamp, 1, 4) AS year,
       SUBSTR(timestamp, 6, 2) AS month,
       SUBSTR(timestamp, 9, 2) AS day
       FROM self;
       '''))
df = df.drop('timestamp')


In [5]:
df.head()

open,high,low,close,volume,rv,year,month,day
f64,f64,f64,f64,f64,f64,str,str,str
3592.47,3597.46,3535.23,3535.23,6.6101e9,0.001254,"""2010""","""01""","""05"""
3545.79,3577.34,3497.66,3564.04,8.5611e9,0.00198,"""2010""","""01""","""06"""
3556.98,3588.77,3541.28,3541.73,7.8250e9,0.001687,"""2010""","""01""","""07"""
3542.44,3558.56,3453.4,3471.46,8.0187e9,0.002066,"""2010""","""01""","""08"""
3453.45,3482.04,3426.82,3480.13,6.0607e9,0.002053,"""2010""","""01""","""09"""


In [6]:
data_x = df.drop('rv').to_numpy()
data_y = df.select('rv').to_numpy()

In [7]:
index = 0
seq_len = 10
data_x[index:index+seq_len]

array([[3592.47, 3597.46, 3535.23, 3535.23, 6610108000.0, '2010', '01',
        '05'],
       [3545.79, 3577.34, 3497.66, 3564.04, 8561099900.0, '2010', '01',
        '06'],
       [3556.98, 3588.77, 3541.28, 3541.73, 7824973800.0, '2010', '01',
        '07'],
       [3542.44, 3558.56, 3453.4, 3471.46, 8018689800.0, '2010', '01',
        '08'],
       [3453.45, 3482.04, 3426.82, 3480.13, 6060716800.0, '2010', '01',
        '09'],
       [3591.31, 3593.68, 3465.56, 3482.05, 8706400700.0, '2010', '01',
        '12'],
       [3477.24, 3535.41, 3437.67, 3534.92, 9343414800.0, '2010', '01',
        '13'],
       [3446.32, 3489.94, 3415.91, 3421.14, 11128022900.0, '2010', '01',
        '14'],
       [3436.68, 3470.11, 3411.94, 3469.05, 8308997100.0, '2010', '01',
        '15'],
       [3472.51, 3499.96, 3448.77, 3482.74, 7228468300.0, '2010', '01',
        '16']], dtype=object)

In [9]:
data_x = df.drop('rv').to_numpy().astype(np.float32)
data_y = df.select('rv').to_numpy().astype(np.float32)

In [16]:
max=data_x.max(0)
min=data_x.min(0)

In [26]:
lower = -1
upper = 1

In [36]:
((max - data_x)/(max - min)*(upper - lower)+lower)

array([[ 0.21263862,  0.22421741,  0.18809569, ...,  1.        ,
         1.        ,  0.73333335],
       [ 0.23693717,  0.23477304,  0.20827115, ...,  1.        ,
         1.        ,  0.6666666 ],
       [ 0.23111236,  0.22877645,  0.18484676, ...,  1.        ,
         1.        ,  0.6       ],
       ...,
       [ 0.43045127,  0.43855834,  0.39000046, ..., -1.        ,
        -0.45454544,  0.13333333],
       [ 0.43836868,  0.44328535,  0.39722848, ..., -1.        ,
        -0.45454544, -0.19999999],
       [ 0.42817664,  0.42547905,  0.39508045, ..., -1.        ,
        -0.45454544, -0.26666665]], dtype=float32)

In [45]:
x_arr = np.array([[1,2,3], [1,2,3]])
y_arr = np.array([[4],[5]])

In [46]:
np.concatenate([x_arr, y_arr], axis=1)

array([[1, 2, 3, 4],
       [1, 2, 3, 5]])